In [1]:
import sys
from pathlib import Path
# Make the repo root importable regardless of CWD or machine: walk up from the
# working dir until we hit a repo marker, then put that dir on sys.path.
_start = Path.cwd()
_root = next((p for p in (_start, *_start.parents)
              if (p / ".git").exists() or (p / "setup.py").exists()), _start)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

# MetabTravLR quickstart

Train SpaceTravLR with **harreman metabolites** added as a modulator group — one summed
column per metabolite (over its transporter gene pairs, both orientations) — then read the
learned `beta_metab@<metabolite>` coefficients back out over labeled gene sets to rank
metabolites by effect. We analyze the coefficients directly — **no perturbation**.

Edit the **Config** and **Gene sets** cells, then run top to bottom. Everything is written
under the dataset directory. On Savio, replace `fit(...)` with the `spawn_worker` cell.

In [ ]:
import os, sys
import numpy as np
import pandas as pd
import scanpy as sc

# SpaceTravLR package (src/) + our metab_processing helpers
_here = os.path.dirname(os.path.abspath('.'))
sys.path.append(os.path.join(os.getcwd(), '..', 'src'))
sys.path.append(os.path.join(os.getcwd(), '..'))

from SpaceTravLR.spaceship import SpaceShip
from metab_processing.SpaceTravLR.metab_loader import load_metabolites
from metab_processing.SpaceTravLR import beta_analysis

In [ ]:
# from harreman_summary import select_tcell_metabolites
# select_tcell_metabolites(f'{dataset_dir}/easy_download')

Wrote 76 metabolites -> /global/scratch/users/fosterangus/MetabTravLR/Data/Xenium/Primary_Dermal_Melanoma/easy_download/harreman_outputs/metabolite_selection.yaml


PosixPath('/global/scratch/users/fosterangus/MetabTravLR/Data/Xenium/Primary_Dermal_Melanoma/easy_download/harreman_outputs/metabolite_selection.yaml')

## Config — data dir / dataset selection

Layout assumed: `DATA_DIR / DATASET / {adata.h5ad, easy_download/harreman_outputs/...}`.
Results are written to `DATA_DIR / DATASET / spacetravlr_output`.

In [4]:
from metab_processing.metab_travlr_config import PROJECT_DATA_DIR, DATA_DIR as METAB_DATA_DIR

DATA_DIR = PROJECT_DATA_DIR
DATASET  = 'Primary_Dermal_Melanoma'   # dataset folder under DATA_DIR

CELL_TYPE_SRC = 'leiden_scVI_res_0.5'   # adata.obs column to use as 'cell_type' (the harreman tier annotation)

dataset_dir    = f'{DATA_DIR}/{DATASET}'
adata_path     = f'{dataset_dir}/adata.h5ad'
harreman_dir   = f'{dataset_dir}/easy_download/harreman_outputs'
selection_yaml = f'{harreman_dir}/metabolite_selection.yaml'
outdir         = f'{dataset_dir}/spacetravlr_output'
betadata_dir   = f'{outdir}/betadata'

for p in (adata_path, selection_yaml):
    assert os.path.exists(p), f'missing: {p}'

## Gene sets

The target genes to train and the labels to score metabolites against. `focus_genes` (the
genes actually trained) is the union of all sets. **Edit these lists.** With exactly
`positive`/`negative` labels, the ranking uses `signed = positive − negative`.

In [5]:
GENE_SETS = {
    'positive': ['CD4', 'CD3E', 'IL2RA'],          # e.g. T-cell activity
    'negative': ['CTLA4', 'FOXP3', 'IL10', 'ENTPD1'],  # e.g. exhaustion
}

focus_genes = list(dict.fromkeys(g for genes in GENE_SETS.values() for g in genes))
print(f'{len(focus_genes)} focus genes:', focus_genes)

7 focus genes: ['CD4', 'CD3E', 'IL2RA', 'CTLA4', 'FOXP3', 'IL10', 'ENTPD1']


In [6]:
adata = sc.read_h5ad(adata_path)
adata.obs['cell_type'] = adata.obs[CELL_TYPE_SRC]
adata.layers['raw_count'] = adata.X
adata

AnnData object with n_obs × n_vars = 112551 × 5006
    obs: 'x_centroid', 'y_centroid', 'transcript_counts', 'control_probe_counts', 'genomic_control_counts', 'control_codeword_counts', 'unassigned_codeword_counts', 'deprecated_codeword_counts', 'total_counts', 'cell_area', 'nucleus_area', 'nucleus_count', 'segmentation_method', '_scvi_batch', '_scvi_labels', 'leiden_scVI_res_0.5', 'leiden_scVI_res_2.5', 'leiden_scVI_res_2', 'leiden_scVI_res_1.5', 'leiden_scVI_res_1', 'leiden_scVI_res_0.75', 'leiden_scVI_res_0.65', 'leiden_scVI_res_0.375', 'leiden_scVI_res_0.25', 'leiden_scVI_res_0.1', 'leiden_scVI_res_0.05', 'cd8', 'cd4', 't_cell', 'Tier1', 'Tier2', 'Tier3', 'Cytotoxic_CD8_score', 'Exhausted_CD8_score', 'Treg_score', 'sub_cluster_5_res_1', 'sub_cluster_5_res_0.75', 'sub_cluster_5_res_0.5', 'sub_cluster_5_res_0.37', 'sub_cluster_5_res_0.25', 'sub_cluster_5_res_0.15', 'sub_cluster_5_res_0.1', 'sub_cluster_5_res_0.05', 'sub_cluster_2_res_1', 'sub_cluster_2_res_0.75', 'sub_cluster_2_res_0

## Metabolites from harreman

`metabolite_selection.yaml` → the `metabolites` dict `{name: [(export, import), ...]}` the
model consumes: per-metabolite orientation expansion (homotypic once, heterotypic both
orientations), filtered to genes in the panel, with metabolites that share an identical
transporter-pair set **merged** into one `nameA|nameB` column. `selection` keeps the original
metabolite→pairs grouping for reference.

In [ ]:
metabolites, selection = load_metabolites(selection_yaml, var_names=adata.var_names)
print(f'{len(selection)} metabolites -> {len(metabolites)} model columns (merged where pair-sets match)')
list(metabolites.items())[:4]

## Setup + train

COMMOT is skipped — harreman is our metabolite prior. Only `focus_genes` are trained.

In [12]:
spacetravlr = SpaceShip(
    name=DATASET.replace('/', '_'),
    outdir=outdir,
    genes=focus_genes,
)

In [13]:
spacetravlr.setup_(adata, overwrite=False, run_commot=False)
assert spacetravlr.is_everything_ok()

AssertionError: Launch script not found

In [ ]:
# Local / single-process training. On Savio use the spawn_worker cell below instead.
spacetravlr.fit(metabolites=metabolites)

In [ ]:
# --- Savio: run this cell (multiple times) to spawn parallel SLURM workers instead of fit() ---
# spacetravlr.focus_genes = focus_genes
# spacetravlr.spawn_worker(
#     account='fc_wagnerlabfca',
#     partition='savio4_gpu',
#     qos='a5k_gpu4_normal',
#     gres='gpu:A5000:1',
#     job_name='MetabTravLR',
#     cpus_per_task=4,
#     lifespan=0.5,
#     python_path='/global/home/users/fosterangus/.conda/envs/spacetravlr/bin/python',
#     metabolites=metabolites,   # if driving via a launch.py, pass metabolites to run_spacetravlr
# )

## Read the metabolite coefficients back out

For each tier (an `adata.obs` cell-type column), group cells by their label and take the
mean/std of every metabolite beta. Written to `easy_download/metabtravlr_outputs/<tier>/`.

In [ ]:
TIERS = ['Tier1', 'Tier2', 'Tier3']
metab_outdir = f'{dataset_dir}/easy_download/metabtravlr_outputs'

# <tier>/metabolites.csv: mean/std per (gene, metabolite, cell type)
metabolite_betas = beta_analysis.write_metabolites(betadata_dir, adata.obs, TIERS, metab_outdir)
metabolite_betas['Tier1'].head()

In [ ]:
# <tier>/histograms.csv: distribution of those mean betas, per modulator group
# (metab / lr / ltf / tf). plot=True also writes histograms.png.
hists = beta_analysis.write_histograms(betadata_dir, adata.obs, TIERS, metab_outdir, genes=focus_genes, plot=True)
hists['Tier1'].head()

,group,left,right,count
0,lr,-0.008692,-0.008408,1
1,lr,-0.008408,-0.008123,0
2,lr,-0.008123,-0.007839,1
3,lr,-0.007839,-0.007554,0
4,lr,-0.007554,-0.007269,0


In [8]:
# Per-cell betas back onto the cells: adata.obsm['beta_<gene>'], names in adata.uns['beta_modulators']
beta_analysis.betas_to_adata(adata, betadata_dir, genes=focus_genes)
adata.obsm['beta_CD4'].shape, adata.uns['beta_modulators']['CD4'][:5]

((112551, 100),
 ['ABCA1@ABCA1',
  'ATP7A@ATP7A',
  'SLC16A4@SLCO2B1',
  'SLCO2B1@SLC16A4',
  'SLCO2B1@SLCO2B1'])

In [ ]:
# 1. Pick a metabolite modulator to plot (names are 'metab@<metabolite>', merged as
#    'metab@<nameA>|<nameB>'). Grab the first one for CD4 as an example.
modulator_name = adata.uns['beta_modulators']['CD4'][0]
modulator_idx = adata.uns['beta_modulators']['CD4'].index(modulator_name)

# 2. Extract that specific column and assign it to a new column in adata.obs
adata.obs[f'beta_CD4_{modulator_name}'] = adata.obsm['beta_CD4'][:, modulator_idx]

# 3. Plot using the standard AnnData object
sc.pl.spatial(
    adata, 
    color=f'beta_CD4_{modulator_name}', 
    spot_size=5, 
    title=f'{DATASET} - CD4 Beta ({modulator_name})'
)

In [8]:
from Harreman.copy_easy_download import save_easy_downloads
save_easy_downloads()